# CSET485 AI and Society - Assignment 1 (Roll 0769 -> Adult Census Income, random_state=769)
Every number in the PDF report is produced by this notebook (run top to bottom).

In [1]:
import json, warnings, os
import numpy as np, pandas as pd, statsmodels.api as sm
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore")
SEED = 769
R = {}   # all results land here -> results.json
os.makedirs("figs", exist_ok=True)

## 0. Data loading and preprocessing
Source: UCI Adult (adult.data + adult.test = 48,842 rows). UCI host was unreachable from my sandbox, so the identical
data is read from a GitHub mirror (jbrownlee/Datasets). A local copy `adult-all.csv` is shipped with the notebook.

In [2]:
cols = ['age','workclass','fnlwgt','education','education_num','marital_status','occupation','relationship',
        'race','sex','capital_gain','capital_loss','hours_per_week','native_country','income']
URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/adult-all.csv"
src = "adult-all.csv" if os.path.exists("adult-all.csv") else URL
raw = pd.read_csv(src, header=None, names=cols, skipinitialspace=True, na_values='?')
R['n_rows'] = len(raw)
R['missing'] = {c:int(v) for c,v in raw.isna().sum().items() if v>0}
R['missing_pct'] = {c:round(100*v/len(raw),2) for c,v in raw.isna().sum().items() if v>0}
R['duplicates'] = int(raw.duplicated().sum())
R['base_rate'] = float((raw.income=='>50K').mean())
R['cap_gain_99999'] = int((raw.capital_gain==99999).sum())
R['cap_gain_nonzero_pct'] = float((raw.capital_gain>0).mean()*100)
R['cap_loss_nonzero_pct'] = float((raw.capital_loss>0).mean()*100)
print(R['n_rows'], R['missing'], R['base_rate'])

48842 {'workclass': 2799, 'occupation': 2809, 'native_country': 857} 0.23928176569346055


In [3]:
df = raw.copy()
# (1) Missing values: '?' only in workclass, occupation, native_country -> keep rows, encode as own category 'Unknown'
for c in ['workclass','occupation','native_country']:
    df[c] = df[c].fillna('Unknown')
# (2) Redundant / non-predictive columns: education (== education_num), fnlwgt (survey sampling weight), relationship
#     (collinear with marital_status + sex; dropped to keep coefficients interpretable)
df = df.drop(columns=['education','fnlwgt','relationship'])
# (3) Outliers: capital_gain / capital_loss are heavily skewed (99999 is a top-code) -> log1p; hours winsorised at 1st/99th pct
df['capital_gain'] = np.log1p(df.capital_gain); df['capital_loss'] = np.log1p(df.capital_loss)
lo, hi = df.hours_per_week.quantile([.01,.99]); R['hours_clip'] = [float(lo), float(hi)]
df['hours'] = df.hours_per_week.clip(lo, hi); df = df.drop(columns='hours_per_week')
# (4) Target and binary encodings
df['y'] = (df.income=='>50K').astype(int); df = df.drop(columns='income')
df['male'] = (df.sex=='Male').astype(int)
df['married'] = (df.marital_status=='Married-civ-spouse').astype(int)
# native_country: 40+ levels, 90% US -> US vs other vs unknown
df['native_country'] = np.where(df.native_country=='United-States','US',np.where(df.native_country=='Unknown','Unknown','Other'))
num_cols = ['age','education_num','capital_gain','capital_loss','hours']
cat_cols = ['workclass','marital_status','occupation','race','native_country']
def design(d, drop=()):
    X = pd.get_dummies(d[num_cols+cat_cols+['male']], columns=cat_cols, drop_first=True, dtype=float)
    X = X.drop(columns=[c for c in X.columns if any(c==x or c.startswith(x+'_') for x in drop)])
    return X
Xall = design(df); yall = df.y
print(Xall.shape)

(48842, 40)


In [4]:
# (5) Train/test split 70/30, random_state = 769, stratified on y. Scaling: StandardScaler fitted on TRAIN only (numeric cols).
train_idx, test_idx = train_test_split(df.index, test_size=0.30, random_state=SEED, stratify=df.y)
Xtr, Xte = Xall.loc[train_idx].copy(), Xall.loc[test_idx].copy()
ytr, yte = yall.loc[train_idx], yall.loc[test_idx]
sc = StandardScaler().fit(Xtr[num_cols])
Xtr[num_cols] = sc.transform(Xtr[num_cols]); Xte[num_cols] = sc.transform(Xte[num_cols])
R['n_train'], R['n_test'], R['n_features'] = len(Xtr), len(Xte), Xtr.shape[1]
R['test_base_rate'] = float(yte.mean())
Xtr_c, Xte_c = sm.add_constant(Xtr, has_constant='add'), sm.add_constant(Xte, has_constant='add')

## Part 1 - Build: Model A (OLS on 0/1 income) vs Model B (logistic)

In [5]:
ols = LinearRegression().fit(Xtr, ytr)
pA = ols.predict(Xte)
med = float(np.median(pA))
highA = (pA >= med).astype(int)
logit = sm.Logit(ytr, Xtr_c).fit(disp=0, maxiter=500)
pB = logit.predict(Xte_c).values
highB = (pB >= 0.5).astype(int)
ct = pd.crosstab(pd.Series(highA,name='A'), pd.Series(highB,name='B'))
cm = [[int(ct.loc[a,b]) if (a in ct.index and b in ct.columns) else 0 for b in (0,1)] for a in (0,1)]
mism = cm[0][1]+cm[1][0]
R['p1'] = dict(median_A=med, mean_A=float(pA.mean()), min_A=float(pA.min()), max_A=float(pA.max()),
    frac_A_below0=float((pA<0).mean()*100), frac_A_above1=float((pA>1).mean()*100),
    cm=cm, mismatch=int(mism), mismatch_pct=float(100*mism/len(yte)),
    flagged_A=int(highA.sum()), flagged_B=int(highB.sum()),
    flagged_A_pct=float(100*highA.mean()), flagged_B_pct=float(100*highB.mean()),
    acc_A=float((highA==yte.values).mean()), acc_B=float((highB==yte.values).mean()),
    prec_A=float(yte.values[highA==1].mean()), prec_B=float(yte.values[highB==1].mean()),
    rec_A=float(highA[yte.values==1].mean()), rec_B=float(highB[yte.values==1].mean()),
    pB_median=float(np.median(pB)), pB_quantile_of_0_5=float((pB<0.5).mean()*100),
    pA_at_B_cut=float(np.mean(pA>=0.5)*100), r2_A=float(ols.score(Xtr,ytr)),
    mismatch_A_only=int(cm[1][0]), mismatch_B_only=int(cm[0][1]),
    corr=float(np.corrcoef(pA,pB)[0,1]))
# also: OLS cut at 0.5 for reference
R['p1']['A_cut05_flagged_pct'] = float((pA>=0.5).mean()*100)
R['p1']['A_cut05_vs_B_mismatch_pct'] = float(((pA>=0.5).astype(int)!=highB).mean()*100)
print(json.dumps(R['p1'],indent=1))

{
 "median_A": 0.22987684871785086,
 "mean_A": 0.2397005732435877,
 "min_A": -0.3711395316455773,
 "max_A": 1.259699272669808,
 "frac_A_below0": 21.429058895789257,
 "frac_A_above1": 0.4572442503241657,
 "cm": [
  [
   7326,
   0
  ],
  [
   4482,
   2845
  ]
 ],
 "mismatch": 4482,
 "mismatch_pct": 30.587592984371803,
 "flagged_A": 7327,
 "flagged_B": 2845,
 "flagged_A_pct": 50.003412270524805,
 "flagged_B_pct": 19.415819286153006,
 "acc_A": 0.713028048863714,
 "acc_B": 0.8432402920903569,
 "prec_A": 0.4522997133888358,
 "prec_B": 0.7124780316344465,
 "rec_A": 0.9452367370222475,
 "rec_B": 0.5781517398745009,
 "pB_median": 0.10966399770011354,
 "pB_quantile_of_0_5": 80.584180713847,
 "pA_at_B_cut": 17.15007165768102,
 "r2_A": 0.3728901564543262,
 "mismatch_A_only": 4482,
 "mismatch_B_only": 0,
 "corr": 0.9385510611402735,
 "A_cut05_flagged_pct": 17.15007165768102,
 "A_cut05_vs_B_mismatch_pct": 3.2894287859141476
}


## Part 2.1 - Simpson's paradox hunt
Split variable: marital status (married-civ-spouse vs everyone else). Key predictor: `male` (sex).
Logistic regression with the same controls, fitted on all 48,842 rows (inference task, not prediction).

In [6]:
def fit_logit(d, drop=()):
    X = design(d, drop); X = X.loc[:, X.std()>0]
    X = sm.add_constant(X, has_constant='add')
    return sm.Logit(d.y, X).fit(disp=0, maxiter=500)
def coef_row(m, v):
    ci = m.conf_int().loc[v]
    return dict(coef=float(m.params[v]), lo=float(ci[0]), hi=float(ci[1]), p=float(m.pvalues[v]), odds=float(np.exp(m.params[v])))
m_full = fit_logit(df)
m_mar  = fit_logit(df[df.married==1], drop=('marital_status',))
m_unm  = fit_logit(df[df.married==0], drop=('marital_status',))
R['p21'] = {'n_full':len(df),'n_married':int((df.married==1).sum()),'n_unmarried':int((df.married==0).sum())}
for v in ['male','age','education_num','hours','capital_gain']:
    R['p21'][v] = {'full':coef_row(m_full,v),'married':coef_row(m_mar,v),'unmarried':coef_row(m_unm,v)}
R['p21']['crude'] = dict(
    male_rate=float(df[df.male==1].y.mean()), female_rate=float(df[df.male==0].y.mean()),
    male_married=float(df[(df.male==1)&(df.married==1)].y.mean()), female_married=float(df[(df.male==0)&(df.married==1)].y.mean()),
    male_unmarried=float(df[(df.male==1)&(df.married==0)].y.mean()), female_unmarried=float(df[(df.male==0)&(df.married==0)].y.mean()),
    male_share_married=float(df[df.male==1].married.mean()*100), female_share_married=float(df[df.male==0].married.mean()*100),
    married_rate=float(df[df.married==1].y.mean()), unmarried_rate=float(df[df.married==0].y.mean()))
# interaction test (is the difference significant?)
dfi = df.copy(); dfi['male_x_married'] = dfi.male*dfi.married
Xi = design(dfi); Xi['male_x_married']=dfi.male_x_married; Xi = sm.add_constant(Xi, has_constant='add')
mi = sm.Logit(dfi.y, Xi).fit(disp=0, maxiter=500)
R['p21']['interaction'] = coef_row(mi,'male_x_married')
# sex gap in predicted probability from a model with NO marital info (policy illustration)
print(json.dumps(R['p21'],indent=1))

{
 "n_full": 48842,
 "n_married": 22379,
 "n_unmarried": 26463,
 "male": {
  "full": {
   "coef": 0.12318512626521949,
   "lo": 0.042691717133902116,
   "hi": 0.20367853539653685,
   "p": 0.002704404360944564,
   "odds": 1.1310937967366153
  },
  "married": {
   "coef": -0.3210659565119059,
   "lo": -0.4286548128910373,
   "hi": -0.2134771001327745,
   "p": 4.947999122330238e-09,
   "odds": 0.7253754061807062
  },
  "unmarried": {
   "coef": 0.6223800556571518,
   "lo": 0.49448399652815,
   "hi": 0.7502761147861536,
   "p": 1.4594427300316446e-21,
   "odds": 1.863357663330386
  }
 },
 "age": {
  "full": {
   "coef": 0.02669754146638077,
   "lo": 0.024193426156372187,
   "hi": 0.02920165677638935,
   "p": 5.811435083365883e-97,
   "odds": 1.027057113592042
  },
  "married": {
   "coef": 0.02321861227078076,
   "lo": 0.020371747725192263,
   "hi": 0.026065476816369254,
   "p": 1.6209655701725542e-57,
   "odds": 1.0234902626224556
  },
  "unmarried": {
   "coef": 0.04779436710488323,
   "

## Part 2.2 - Omitted-variable bias (drop `marital_status`)

In [7]:
m_red = fit_logit(df, drop=('marital_status',))
keep = ['age','education_num','hours','capital_gain','capital_loss','male']
rows = []
for v in keep:
    b, r = float(m_full.params[v]), float(m_red.params[v])
    rows.append(dict(var=v, full=b, reduced=r, pct=100*(r-b)/abs(b)))
R['p22'] = {'rows':rows, 'dropped':'marital_status',
            'llf_full':float(m_full.llf),'llf_red':float(m_red.llf),
            'pr2_full':float(m_full.prsquared),'pr2_red':float(m_red.prsquared),
            'aic_full':float(m_full.aic),'aic_red':float(m_red.aic)}
# fit-quality on the held-out test set for both models (same split)
print(pd.DataFrame(rows))
print(m_full.params.filter(like='marital').round(3).to_dict())
R['p22']['marital_coefs'] = {k:float(v) for k,v in m_full.params.filter(like='marital').items()}
# correlation of dropped var with included predictors
R['p22']['corr_married'] = {v: float(df[[v,'married']].corr().iloc[0,1]) for v in ['age','education_num','hours','male','capital_gain']}

             var      full   reduced         pct
0            age  0.026698  0.042003   57.329980
1  education_num  0.287925  0.257343  -10.621404
2          hours  0.031602  0.032916    4.159081
3   capital_gain  0.209858  0.203109   -3.216091
4   capital_loss  0.153257  0.149907   -2.186281
5           male  0.123185  1.130398  817.641992
{'marital_status_Married-AF-spouse': 2.559, 'marital_status_Married-civ-spouse': 2.178, 'marital_status_Married-spouse-absent': 0.208, 'marital_status_Never-married': -0.409, 'marital_status_Separated': -0.077, 'marital_status_Widowed': -0.032}


## Part 2.3 - Adversarial subset (confident and wrong)

In [8]:
te = df.loc[test_idx].copy(); te['p'] = pB; te['pred'] = highB
fp = te[(te.p>=0.75)&(te.y==0)]; fn = te[(te.p<=0.25)&(te.y==1)]
bad = pd.concat([fp.assign(kind='FP'), fn.assign(kind='FN')])
R['p23'] = dict(n_fp=len(fp), n_fn=len(fn), n_bad=len(bad), pct_bad=100*len(bad)/len(te),
                n_conf=int(((te.p>=0.75)|(te.p<=0.25)).sum()))
R['p23']['pct_bad_of_conf'] = 100*len(bad)/R['p23']['n_conf']
R['p23']['n_hi_pos'] = int((te.p>=0.75).sum()); R['p23']['fp_rate_hi'] = 100*len(fp)/max(1,R['p23']['n_hi_pos'])
R['p23']['n_lo'] = int((te.p<=0.25).sum()); R['p23']['fn_rate_lo'] = 100*len(fn)/max(1,R['p23']['n_lo'])
samp = bad.sample(n=min(50,len(bad)), random_state=SEED)
R['p23']['sample_n']=len(samp); R['p23']['sample_fp']=int((samp.kind=='FP').sum()); R['p23']['sample_fn']=int((samp.kind=='FN').sum())
def prof(d):
    return dict(age_mean=float(d.age.mean()), age_median=float(d.age.median()), pct_male=float(d.male.mean()*100),
        pct_married=float(d.married.mean()*100), edu_mean=float(d.education_num.mean()), hours_mean=float(d.hours.mean()),
        pct_cap_gain=float((d.capital_gain>0).mean()*100), pct_cap_loss=float((d.capital_loss>0).mean()*100),
        pct_white=float((d.race=='White').mean()*100), pct_age_ge50=float((d.age>=50).mean()*100),
        pct_age_lt30=float((d.age<30).mean()*100), pct_hours_gt45=float((d.hours>45).mean()*100), pct_hours_lt35=float((d.hours<35).mean()*100),
        pct_prof=float(d.occupation.isin(['Exec-managerial','Prof-specialty']).mean()*100),
        pct_private=float((d.workclass=='Private').mean()*100), pct_bach_plus=float((d.education_num>=13).mean()*100))
R['p23']['prof_sample']=prof(samp); R['p23']['prof_test']=prof(te)
R['p23']['prof_fp']=prof(samp[samp.kind=='FP']) if (samp.kind=='FP').any() else {}
R['p23']['prof_fn']=prof(samp[samp.kind=='FN']) if (samp.kind=='FN').any() else {}
# age non-linearity: positive rate by age band in full data
bands = pd.cut(df.age,[16,25,30,35,40,45,50,55,60,65,100])
R['p23']['age_rate'] = {str(k):float(v) for k,v in df.groupby(bands).y.mean().items()}
# misclassification of the logit by age band & married on the test set
R['p23']['top_occ_fp'] = samp[samp.kind=='FP'].occupation.value_counts().head(3).to_dict()
R['p23']['top_occ_fn'] = samp[samp.kind=='FN'].occupation.value_counts().head(3).to_dict()
R['p23']['top_ms_fn'] = samp[samp.kind=='FN'].marital_status.value_counts().head(3).to_dict()
R['p23']['top_ms_fp'] = samp[samp.kind=='FP'].marital_status.value_counts().head(3).to_dict()
# key mechanism test: married-flag interaction in the full test set
R['p23']['married_rate_in_conf_wrong'] = float(bad.married.mean()*100)
R['p23']['hours_effect_married_vs_not'] = {'married':float(m_mar.params['hours']),'unmarried':float(m_unm.params['hours'])}
samp.to_csv('adversarial_50_rows.csv', index=False)
print(json.dumps(R['p23'],indent=1))

{
 "n_fp": 175,
 "n_fn": 590,
 "n_bad": 765,
 "pct_bad": 5.220773902955027,
 "n_conf": 10774,
 "pct_bad_of_conf": 7.100426953777613,
 "n_hi_pos": 1236,
 "fp_rate_hi": 14.158576051779935,
 "n_lo": 9538,
 "fn_rate_lo": 6.185783183057245,
 "sample_n": 50,
 "sample_fp": 13,
 "sample_fn": 37,
 "prof_sample": {
  "age_mean": 41.06,
  "age_median": 40.0,
  "pct_male": 72.0,
  "pct_married": 74.0,
  "edu_mean": 10.68,
  "hours_mean": 42.5,
  "pct_cap_gain": 6.0,
  "pct_cap_loss": 10.0,
  "pct_white": 88.0,
  "pct_age_ge50": 24.0,
  "pct_age_lt30": 16.0,
  "pct_hours_gt45": 30.0,
  "pct_hours_lt35": 14.000000000000002,
  "pct_prof": 24.0,
  "pct_private": 70.0,
  "pct_bach_plus": 34.0
 },
 "prof_test": {
  "age_mean": 38.82713437521327,
  "age_median": 37.0,
  "pct_male": 67.11936122295775,
  "pct_married": 46.120248413294206,
  "edu_mean": 10.081553265542892,
  "hours_mean": 40.32866989694943,
  "pct_cap_gain": 8.19627380058691,
  "pct_cap_loss": 4.565617962192042,
  "pct_white": 85.6479901726

## Part 2.4 - Fairness trade-off (sensitive variable: sex; Group A = Female, Group B = Male)

In [9]:
ths = np.round(np.arange(0.05,0.96,0.10),2)
A = (te.male==0).values; B = (te.male==1).values; yt = yte.values
def safe(a,b): return float(a/b) if b>0 else float('nan')
rows=[]
for t in ths:
    yh = (pB>=t).astype(int)
    def rates(m):
        y_, p_ = yt[m], yh[m]
        tp=((p_==1)&(y_==1)).sum(); fp_=((p_==1)&(y_==0)).sum(); P=(y_==1).sum(); N=(y_==0).sum()
        return dict(sel=p_.mean(), tpr=safe(tp,P), fpr=safe(fp_,N), ppv=safe(tp,tp+fp_), n_pos=int(p_.sum()))
    ra, rb = rates(A), rates(B)
    dp = safe(ra['sel'], rb['sel']); tprr = safe(ra['tpr'],rb['tpr']); fprr = safe(ra['fpr'],rb['fpr'])
    eo = np.nanmin([tprr,fprr]) if not (np.isnan(tprr) and np.isnan(fprr)) else float('nan')
    pp = safe(ra['ppv'], rb['ppv'])
    rows.append(dict(t=float(t), sel_A=ra['sel'], sel_B=rb['sel'], DP=dp, TPR_A=ra['tpr'], TPR_B=rb['tpr'], FPR_A=ra['fpr'], FPR_B=rb['fpr'],
                     TPRr=tprr, FPRr=fprr, EO=eo, PPV_A=ra['ppv'], PPV_B=rb['ppv'], PP=pp, acc=float((yh==yt).mean()),
                     acc_A=float((yh[A]==yt[A]).mean()), acc_B=float((yh[B]==yt[B]).mean())))
sweep = pd.DataFrame(rows); R['p24']=dict(rows=rows, nA=int(A.sum()), nB=int(B.sum()),
    base_A=float(yt[A].mean()), base_B=float(yt[B].mean()))
print(sweep.round(3).to_string())
fig, ax = plt.subplots(figsize=(6.4,5.6))
ax.plot(sweep.DP, sweep.EO, '-o', color='#1f4e79'); 
for i,(_,r) in enumerate(sweep.iterrows()):
    if not np.isnan(r.DP) and not np.isnan(r.EO): ax.annotate(f"t={r.t:.2f}", (r.DP,r.EO), textcoords='offset points', xytext=((8,8) if i%2==0 else (-34,-14)), fontsize=8)
ax.axvline(1,ls='--',c='grey',lw=.8); ax.axhline(1,ls='--',c='grey',lw=.8); ax.scatter([1],[1],marker='*',s=200,c='gold',edgecolor='k',zorder=5,label='ideal (1,1)')
ax.set_xlim(0,1.5); ax.set_ylim(0,1.5); ax.set_xlabel('Demographic Parity ratio (Female / Male)'); ax.set_ylabel('Equalized Odds ratio min(TPR, FPR ratio)')
ax.set_title('Fairness trade-off across thresholds (labels = threshold)'); ax.legend(loc='lower right'); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig('figs/tradeoff.png', dpi=160); plt.close()
# supporting chart: all three metrics vs threshold
fig, ax = plt.subplots(figsize=(6.4,4)); 
for c,l in [('DP','DP'),('EO','EO'),('PP','PP')]: ax.plot(sweep.t, sweep[c], '-o', label=l)
ax.axhline(1,ls='--',c='grey'); ax.set_ylim(0,1.5); ax.set_xlabel('threshold t'); ax.set_ylabel('ratio (Female/Male)'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig('figs/metrics_vs_t.png', dpi=160); plt.close()
# part-1 figure: OLS prediction histogram
fig, ax = plt.subplots(figsize=(6.4,3.6)); ax.hist(pA, bins=60, color='#7aa6c2'); ax.axvline(med,c='r',label=f'median={med:.3f}'); ax.axvline(0.5,c='k',ls='--',label='0.5'); ax.legend(); ax.set_xlabel('OLS predicted value (test)'); ax.set_ylabel('count')
plt.tight_layout(); plt.savefig('figs/ols_hist.png', dpi=160); plt.close()
# part-2.1 figure: coefficient forest
fig, ax = plt.subplots(figsize=(6.4,3.2))
for i,(k,l) in enumerate([('full','Full'),('married','Married'),('unmarried','Not married')]):
    c=R['p21']['male'][k]; ax.errorbar(c['coef'], i, xerr=[[c['coef']-c['lo']],[c['hi']-c['coef']]], fmt='o', capsize=4)
ax.axvline(0,c='k',lw=.8); ax.set_yticks([0,1,2]); ax.set_yticklabels(['Full','Married','Not married']); ax.set_xlabel("logit coefficient of male (95% CI)"); plt.tight_layout(); plt.savefig('figs/simpson.png',dpi=160); plt.close()

      t  sel_A  sel_B     DP  TPR_A  TPR_B  FPR_A  FPR_B   TPRr   FPRr     EO  PPV_A  PPV_B     PP    acc  acc_A  acc_B
0  0.05  0.422  0.713  0.592  0.952  0.984  0.358  0.594  0.967  0.602  0.602  0.244  0.419  0.582  0.612  0.676  0.581
1  0.15  0.209  0.566  0.369  0.800  0.941  0.137  0.403  0.851  0.341  0.341  0.414  0.504  0.821  0.752  0.856  0.701
2  0.25  0.135  0.454  0.297  0.655  0.863  0.072  0.276  0.759  0.261  0.261  0.525  0.577  0.910  0.810  0.899  0.766
3  0.35  0.097  0.361  0.267  0.545  0.759  0.042  0.188  0.718  0.224  0.224  0.611  0.638  0.957  0.835  0.913  0.796
4  0.45  0.071  0.291  0.244  0.445  0.659  0.026  0.131  0.676  0.197  0.197  0.676  0.686  0.985  0.842  0.917  0.805
5  0.55  0.050  0.230  0.219  0.347  0.554  0.014  0.088  0.627  0.161  0.161  0.748  0.733  1.021  0.841  0.917  0.803
6  0.65  0.033  0.177  0.185  0.248  0.460  0.007  0.054  0.538  0.124  0.124  0.816  0.787  1.038  0.836  0.913  0.798
7  0.75  0.019  0.116  0.162  0.155  0.3

## Supporting statistics used in the write-up

In [10]:
from scipy.stats import binomtest
k = int(samp.married.sum()); p0 = float(te.married.mean())
R['p23']['binom_p_married'] = float(binomtest(k, len(samp), p0).pvalue); R['p23']['n_married_sample']=k
cf = logit.params
R['coefs_train'] = {c: float(cf[c]) for c in ['const','age','education_num','hours','capital_gain','capital_loss','male',
                    'marital_status_Married-civ-spouse','marital_status_Never-married']}
R['train_std'] = dict(zip(num_cols, [float(x) for x in sc.scale_]))
R['p23']['mean_p_fp'] = float(fp.p.mean()); R['p23']['mean_p_fn'] = float(fn.p.mean())
R['p23']['pct_fp_married_all'] = float(fp.married.mean()*100); R['p23']['pct_fn_married_all'] = float(fn.married.mean()*100)
R['p23']['pct_age_ge50_all_bad'] = float((bad.age>=50).mean()*100)
R['p1']['all_B_flagged_also_A'] = bool(((highB==1)&(highA==1)).sum()==highB.sum())
# trade-off quantification (percent change in metrics when moving between thresholds)
sw = sweep.set_index('t')
def pc(m,a,b): return float(100*(sw.loc[b,m]-sw.loc[a,m])/sw.loc[a,m])
t_pp = float(sweep.iloc[(sweep.PP-1).abs().argsort().iloc[0]].t)
R['p24']['best_t'] = {m: float(sweep.iloc[(sweep[m]-1).abs().argsort().iloc[0]].t) for m in ['DP','EO','PP']}
R['p24']['best_acc_t'] = float(sweep.iloc[sweep.acc.argmax()].t)
a_, b_ = 0.45, 0.05
R['p24']['trade'] = dict(frm=a_, to=b_, dp_gain=pc('DP',a_,b_), eo_gain=pc('EO',a_,b_), pp_change=pc('PP',a_,b_), acc_change=pc('acc',a_,b_),
    cost_dp_pp=abs(pc('PP',a_,b_))/pc('DP',a_,b_), cost_eo_pp=abs(pc('PP',a_,b_))/pc('EO',a_,b_))
R['p24']['dp_eo_corr'] = float(sweep[['DP','EO']].corr().iloc[0,1])
R['p24']['base_pos_A'] = int(yt[A].sum()); R['p24']['base_pos_B'] = int(yt[B].sum())
json.dump(R, open('results.json','w'), indent=1, default=float)
print('saved results.json')

saved results.json


## Report generation: every number below is read from `results.json` produced above.

In [11]:
import json, re, math
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import cm as CM
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, PageBreak, KeepTogether
R = json.load(open('results.json'))
p1, p21, p22, p23, p24 = R['p1'], R['p21'], R['p22'], R['p23'], R['p24']
sw = {round(r['t'],2): r for r in p24['rows']}
def f(x, d=3): return 'n/a' if x is None or (isinstance(x,float) and math.isnan(x)) else f"{x:.{d}f}"
def n(x): return f"{int(x):,}"
def ci(c): return f"{c['coef']:+.3f} (95% CI {c['lo']:+.3f} to {c['hi']:+.3f})"
cmx = p1['cm']; male = p21['male']; cr = p21['crude']
sa, sb = p23['prof_sample'], p23['prof_test']; fpP, fnP = p23['prof_fp'], p23['prof_fn']
tr = p24['trade']; ct = R['coefs_train']
rows22 = {r['var']: r for r in p22['rows']}
t35, t05, t45, t15 = sw[0.35], sw[0.05], sw[0.45], sw[0.15]

ANS = {}
ANS[1] = f"""The two rules agree on {n(cmx[0][0]+cmx[1][1])} of the {n(R['n_test'])} test individuals and disagree on {n(p1['mismatch'])} of them ({p1['mismatch_pct']:.2f}%). All disagreements run in one direction: {n(cmx[1][0])} people are "high risk" under Model A's median rule but "low risk" under Model B, while {cmx[0][1]} people are flagged by Model B alone. So the two models do not disagree about who is relatively riskier; their predicted scores correlate at r = {p1['corr']:.3f}. They disagree about where the line is drawn. The median of the OLS predictions is {p1['median_A']:.3f}, so by construction 50.0% of the test set ({n(p1['flagged_A'])} people) is flagged. The logistic cut-off of 0.5 flags only {p1['flagged_B_pct']:.1f}% ({n(p1['flagged_B'])}), because {p1['pB_quantile_of_0_5']:.1f}% of test cases have a predicted probability below 0.5 and the median logistic probability is just {p1['pB_median']:.3f}. The true share of >$50K earners in the test set is {R['test_base_rate']*100:.1f}%, so the median rule must flag roughly twice as many people as are truly positive. That is why its precision is {p1['prec_A']*100:.1f}% against {p1['prec_B']*100:.1f}% for Model B, while its recall is {p1['rec_A']*100:.1f}% against {p1['rec_B']*100:.1f}%. A second cause is structural: OLS on a 0/1 outcome is a linear probability model, so its outputs are not probabilities. {p1['frac_A_below0']:.1f}% of its predictions are negative (minimum {p1['min_A']:.3f}) and {p1['frac_A_above1']:.2f}% exceed 1 (maximum {p1['max_A']:.3f}), whereas the sigmoid keeps logistic outputs inside (0, 1). When I cut OLS at 0.5 instead of its median, it flags {p1['A_cut05_flagged_pct']:.1f}% and mismatches Model B on only {p1['A_cut05_vs_B_mismatch_pct']:.2f}% of cases. The {p1['mismatch_pct']:.2f}% mismatch is therefore almost entirely a threshold-definition effect (a relative quantile rule versus an absolute probability rule), not a difference in learned signal."""

ANS[2] = f"""I split the data by marital status (married-civ-spouse, n = {n(p21['n_married'])}, versus all others, n = {n(p21['n_unmarried'])}) and tracked the coefficient on male, with identical controls (age, education, hours, capital gain and loss, workclass, occupation, race, country). In the full model it is {ci(male['full'])}, an odds ratio of {male['full']['odds']:.2f}: a small male advantage. Among married people it reverses to {ci(male['married'])}, odds ratio {male['married']['odds']:.2f}. Among everyone else it is {ci(male['unmarried'])}, odds ratio {male['unmarried']['odds']:.2f}. None of the three intervals overlap, and a male-by-married interaction term confirms it at {p21['interaction']['coef']:+.3f} (CI {p21['interaction']['lo']:+.3f} to {p21['interaction']['hi']:+.3f}). The raw rates show why: married women and men earn >$50K at {cr['female_married']*100:.1f}% and {cr['male_married']*100:.1f}% (essentially equal), while among the unmarried it is {cr['female_unmarried']*100:.1f}% versus {cr['male_unmarried']*100:.1f}%. The pooled result is a composition effect. Marriage is the strongest correlate of high income ({cr['married_rate']*100:.1f}% versus {cr['unmarried_rate']*100:.1f}%), and {cr['male_share_married']:.1f}% of men are in that group against {cr['female_share_married']:.1f}% of women, so much of the overall male advantage of {cr['male_rate']*100:.1f}% versus {cr['female_rate']*100:.1f}% flows through marital status. The two opposite subgroup effects partly cancel in the pooled coefficient. If the subgroup were ignored, the pooled model would suggest a mild, uniform male advantage and a policymaker would likely conclude that sex is a minor factor. The subgroup fits show the opposite: sex matters a great deal, in opposite directions. A single sex-based correction would under-serve unmarried women, whose odds are {1/male['unmarried']['odds']:.2f} times those of unmarried men, and would over-correct for married women, whose measured odds already match or exceed those of married men once age, education and hours are held fixed. Subgroup analysis is therefore a precondition for any policy conclusion here, not an optional robustness check."""

ANS[3] = f"""I dropped marital_status, the six dummies led by Married-civ-spouse, whose coefficient in the full model is {R['p22']['marital_coefs']['marital_status_Married-civ-spouse']:+.3f}. I chose it because it is correlated with the other predictors (r with male = {p22['corr_married']['male']:.2f}, with age = {p22['corr_married']['age']:.2f}, with hours = {p22['corr_married']['hours']:.2f}) and strongly related to the outcome. The table in Part 2.2 shows the shifts. Age moved from {rows22['age']['full']:.4f} to {rows22['age']['reduced']:.4f} ({rows22['age']['pct']:+.1f}%), education_num from {rows22['education_num']['full']:.4f} to {rows22['education_num']['reduced']:.4f} ({rows22['education_num']['pct']:+.1f}%), hours from {rows22['hours']['full']:.4f} to {rows22['hours']['reduced']:.4f} ({rows22['hours']['pct']:+.1f}%), and capital_gain from {rows22['capital_gain']['full']:.4f} to {rows22['capital_gain']['reduced']:.4f} ({rows22['capital_gain']['pct']:+.1f}%). The largest shift by far is male, from {rows22['male']['full']:+.4f} to {rows22['male']['reduced']:+.4f}, a change of {rows22['male']['pct']:+.0f}%: the odds ratio jumps from {math.exp(rows22['male']['full']):.2f} to {math.exp(rows22['male']['reduced']):.2f}. Without marital status, the model attributes to sex what actually belongs to marriage, because men are far more often in the married group. Pseudo-R-squared falls from {p22['pr2_full']:.3f} to {p22['pr2_red']:.3f} and AIC rises by {p22['aic_red']-p22['aic_full']:,.0f}, so the reduced model fits clearly worse, but nothing in its own output warns that its sex coefficient is wrong. The reduced model is not obviously broken: it still converges, every p-value is tiny, and most coefficients move by less than 11%. A reader who saw only that table would report a sex effect nine times too large. The lesson is that the stability of a coefficient across plausible specifications is a result in its own right. A model I have not stress-tested this way can look statistically sound while its most policy-relevant coefficient is wrong by an order of magnitude. In practice I would never publish a single specification of this model without showing how its key coefficients move when plausible controls are added or removed."""

ANS[4] = f"""The omitted-variable test above only worked because I already knew which variable to drop. For a confound I cannot see, I would run a three-step audit that combines a domain expert with external data. First, residual analysis on held-out data: my model makes {n(p23['n_bad'])} confident errors ({p23['pct_bad']:.2f}% of the test set), and those errors are not random, since {sa['pct_married']:.0f}% of my 50-row sample is married versus {sb['pct_married']:.1f}% of the test set. Errors that cluster on an observed variable indicate that something correlated with it is missing, so I would fit a shallow decision tree to the residuals and read off the splits. Second, a domain-expert audit: a labour economist would list what drives income in the real world but is absent here, such as years of experience, industry, employer size, spouse's income and region, and I would mark each as plausibly correlated with both sex and income. Third, external validation: I would link to a richer source such as American Community Survey microdata, which records many of these variables, add the candidates one at a time and watch how the sex coefficient moves. In my own data, one omission moved it by {rows22['male']['pct']:+.0f}%, so I would set a pre-registered tolerance, say a shift of more than 25% triggers redesign. Where no external data exist, I would use Oster-style coefficient-stability bounds, which ask how strong an unobserved variable would have to be to erase the effect. If the model fails on a different year, state or population, that is further evidence of a missing driver. None of this proves that a confound is absent, but it turns "I cannot see it" into a measurable sensitivity range that can be documented before deployment."""

ANS[5] = f"""The test set contains {n(p23['n_bad'])} high-confidence errors ({p23['pct_bad']:.2f}% of {n(R['n_test'])}): {p23['n_fp']} false positives (p at least 0.75, actual 0) and {p23['n_fn']} false negatives (p at most 0.25, actual 1). I sampled 50 ({p23['sample_fp']} FP, {p23['sample_fn']} FN). The shared trait is marriage: {sa['pct_married']:.0f}% ({p23['n_married_sample']} of 50) are married-civ-spouse versus {sb['pct_married']:.1f}% of the test set (binomial p = {p23['binom_p_married']:.1e}), including 100% of the false positives. The false positives are also 92.3% male, work {fpP['hours_mean']:.1f} hours a week on average, hold {fpP['pct_bach_plus']:.0f}% bachelor's degrees or higher, and {fpP['pct_age_ge50']:.0f}% are 50 or older, against {sb['pct_age_ge50']:.1f}% in the test set. The failure mechanism is additivity. In the training model, married-civ-spouse adds {ct['marital_status_Married-civ-spouse']:+.2f} to the log-odds, a roughly {math.exp(ct['marital_status_Married-civ-spouse']):.0f}-fold increase in odds, and education adds {ct['education_num']:+.2f} per standard deviation. For a married, educated man working long hours these terms stack to p of at least 0.75, but the effects are not constant: in Part 2.1 the hours slope is {p21['hours']['married']['coef']:.3f} for married versus {p21['hours']['unmarried']['coef']:.3f} for others, and the sex coefficient flips sign. The age term is also linear ({ct['age']:+.2f} per SD) while the share of >$50K earners rises from {list(p23['age_rate'].values())[0]*100:.1f}% (age 25 or under) to {max(p23['age_rate'].values())*100:.1f}% (45 to 50) and falls to {list(p23['age_rate'].values())[-1]*100:.1f}% (over 65), so people past their peak are over-scored. The false negatives are the mirror image: {fnP['pct_married']:.0f}% of them are married, and craft-repair and machine-operator jobs lead the occupations. Their mean predicted probability is only {p23['mean_p_fn']:.2f}, because low education and short hours pull the additive score down, yet they still earn above $50K, a combination the score cannot represent. The statistical causes are missing interactions (married by sex, hours and education) and a missing age-squared term, so the model is confidently wrong on exactly the people for whom its single linear boundary is a poor approximation."""

ANS[6] = f"""Take two people with identical true outcomes, a woman and a man who both really earn above $50K. At t = 0.35 the model correctly identifies {t35['TPR_A']*100:.1f}% of such women but {t35['TPR_B']*100:.1f}% of such men (TPR ratio {t35['TPRr']:.2f}). So the woman has about a one-in-two chance of being recognised and the man about three in four. Selection rates are {t35['sel_A']*100:.1f}% for women and {t35['sel_B']*100:.1f}% for men, a demographic parity of {t35['DP']:.2f}. At this threshold predictive parity is nearly perfect ({t35['PP']:.3f}): a flagged woman has a {t35['PPV_A']*100:.1f}% chance of truly being high-income and a flagged man {t35['PPV_B']*100:.1f}%. Now lower the threshold to 0.05 to chase parity. DP rises to {t05['DP']:.2f} and EO to {t05['EO']:.2f}, and her chance of recognition rises to {t05['TPR_A']*100:.1f}%. But the cost is large. The model now flags {t05['sel_B']*100:.1f}% of all men and {t05['sel_A']*100:.1f}% of women, and the false-positive rate is {t05['FPR_B']*100:.1f}% for men and {t05['FPR_A']*100:.1f}% for women. A flagged woman is now truly high-income only {t05['PPV_A']*100:.1f}% of the time (a flagged man {t05['PPV_B']*100:.1f}%), PP falls to {t05['PP']:.2f} (a {abs(tr['pp_change']):.0f}% drop from {t45['PP']:.3f} at t = 0.45), and overall accuracy drops from {t45['acc']*100:.1f}% to {t05['acc']*100:.1f}%. Flagged women now carry the burden of a weaker signal, and any institution acting on the flag, whether a lender or a screener, makes about three wrong decisions in four about the women it selects. In other words, the choice of threshold redistributes error between the two groups rather than removing it. Prioritising parity of selection therefore trades one group's recognition rate against the reliability of the flag for the same group."""

ANS[7] = f"""I choose t = 0.35. The first number is predictive parity: PP = {t35['PP']:.3f}, with precision of {t35['PPV_A']*100:.1f}% for women and {t35['PPV_B']*100:.1f}% for men. A positive flag means almost the same thing for both groups, which matters when a human or institution acts on the flag. The second is accuracy: {t35['acc']*100:.1f}% overall, within {(sw[0.45]['acc']-t35['acc'])*100:.1f} points of the best sweep value ({t45['acc']*100:.1f}% at t = 0.45), and {t35['acc_A']*100:.1f}% for women and {t35['acc_B']*100:.1f}% for men, so no group pays for the choice through noticeably worse accuracy. The third is that at t = 0.35 the false-positive ratio is {t35['FPRr']:.2f}: the model wrongly flags {t35['FPR_A']*100:.1f}% of low-income women against {t35['FPR_B']*100:.1f}% of low-income men, so women are not being wrongly pushed into the favourable class more often. The strongest counterargument is that DP = {t35['DP']:.2f} and EO = {t35['EO']:.2f} are far from 1. Selection is {t35['sel_A']*100:.1f}% for women versus {t35['sel_B']*100:.1f}% for men, and the true-positive ratio is {t35['TPRr']:.2f}, so a qualified woman is about {(1-t35['TPRr'])*100:.0f}% less likely to be recognised than a qualified man. I accept this critique as valid: precision parity can coexist with a real opportunity gap, and the threshold alone cannot fix it because the base rates differ ({p24['base_A']*100:.1f}% of women versus {p24['base_B']*100:.1f}% of men in the test set). My choice is defensible as a calibration-first position, not as a claim that the outcome is fair. It would need to be paired with human review of borderline women, monitoring of the TPR gap over time, and a commitment to revisit the threshold if the recognition gap persists after deployment."""

ANS[8] = f"""I would recommend prioritising equalized odds, more precisely equality of true-positive rates (equal opportunity), over demographic parity. The reason is that base rates in the test set differ sharply ({p24['base_A']*100:.1f}% of women versus {p24['base_B']*100:.1f}% of men are >$50K), so forcing equal selection rates would require the model to ignore real differences in the label distribution or apply sex-specific cut-offs. EO conditions on the true outcome, so it targets the harm I care about, which is qualified people being missed. My numbers show the gap clearly: at t = 0.35 the TPR ratio is {t35['TPRr']:.2f}, whereas DP is {t35['DP']:.2f}. In this dataset the sweep shows no sharp DP-versus-EO trade-off, because the two move together (correlation {p24['dp_eo_corr']:.2f}) and both peak at t = 0.05 (DP {t05['DP']:.2f}, EO {t05['EO']:.2f}). The real conflict is with predictive parity and accuracy. Moving from t = 0.45 to t = 0.05 lifts DP by {tr['dp_gain']:.0f}% and EO by {tr['eo_gain']:.0f}% but cuts PP by {abs(tr['pp_change']):.0f}% (from {t45['PP']:.3f} to {t05['PP']:.3f}) and accuracy by {abs(tr['acc_change']):.0f}%, a trade-off cost of {tr['cost_dp_pp']:.2f} units of PP per unit of DP gained. Even the best EO ({t05['EO']:.2f}) is far below 1, so no threshold delivers it. I also flag a limitation of the definition: min(TPR ratio, FPR ratio) is dominated by the FPR ratio ({t35['FPRr']:.2f} at t = 0.35), which reflects women being wrongly flagged less often, a gap that is not a harm. I would therefore report the TPR ratio separately. What I sacrifice is DP, which stays near {t35['DP']:.2f}. I accept that because the selection gap partly reflects different base rates the model cannot change, and I would address the underlying disparity with interventions outside the model rather than with quotas hidden in a threshold."""

P4_DIRECT = f"""I would recommend conditional deployment only: as decision support with human review, not as an automated decision-maker. The headline reason is that the model is {p1['acc_B']*100:.1f}% accurate overall at t = 0.5, yet at t = 0.35 it recognises only {t35['TPR_A']*100:.1f}% of qualified women against {t35['TPR_B']*100:.1f}% of qualified men at t = 0.35."""
P4_WHY = f"""A single number fails for several reasons that I found myself. First, overall accuracy hides subgroup gaps. The {p1['acc_B']*100:.1f}% headline says nothing about the fact that qualified women are recognised far less often than qualified men, and accuracy is also inflated by the {(1-R['test_base_rate'])*100:.0f}% of people who are simply negative, so a lazy model that flags nobody would already score {(1-R['test_base_rate'])*100:.1f}%. Second, the threshold is a values question, not a data-driven fact. Moving t from 0.45 to 0.05 raises DP by {tr['dp_gain']:.0f}% but cuts accuracy by {abs(tr['acc_change']):.0f}%. The data can describe both options but cannot tell a policymaker which harm is more acceptable. Third, fairness metrics conflict. At t = 0.45 predictive parity is nearly perfect ({t45['PP']:.3f}) while DP is only {t45['DP']:.2f}; at t = 0.05 DP is best but PP collapses to {t05['PP']:.2f}. Choosing a metric is choosing whom to protect. Fourth, the model fails confidently on predictable subsets. {p23['n_bad']} test cases ({p23['pct_bad']:.1f}%) are confidently wrong, and {sa['pct_married']:.0f}% of my 50-row sample are married people, because the additive logistic form cannot capture interactions. A single score gives no hint of where those failures are concentrated. Fifth, hidden confounds remain unknowable. Dropping a single variable, marital status, moved the sex coefficient by {rows22['male']['pct']:+.0f}%, and the reduced model gave no warning. If I had not thought of that variable, my headline sex effect would have been wrong by an order of magnitude. Sixth, the subgroup structure can reverse the story: the sex coefficient is {male['full']['coef']:+.2f} overall but {male['married']['coef']:+.2f} among married people and {male['unmarried']['coef']:+.2f} among the unmarried, so even an apparently small aggregate effect can conceal large opposing effects. Finally, the label is itself a limitation: income above $50K measured decades ago is a proxy for what we care about, not the thing itself. Asking for one number also hides who bears the errors: in my data a false negative means a qualified person is overlooked, a false positive means an unqualified person is selected, and these fall on different people at different rates depending on the threshold. Each of these facts could be reduced to a number, but no single number carries all of them at once."""
P4_EVID = f"""Evidence from my own analysis: accuracy {p1['acc_B']*100:.1f}% versus TPR {t35['TPR_A']*100:.1f}%/{t35['TPR_B']*100:.1f}% (women/men, t = 0.35); DP {t45['DP']:.2f} versus PP {t45['PP']:.3f} at t = 0.45; {p23['n_bad']} confident errors ({p23['pct_bad']:.2f}%); {rows22['male']['pct']:+.0f}% sex-coefficient shift on omitting marital status; sex coefficient {male['full']['coef']:+.2f}, {male['married']['coef']:+.2f} and {male['unmarried']['coef']:+.2f} in the full, married and unmarried groups; Model A versus B flagged {p1['mismatch_pct']:.1f}% of people differently."""
P4_CLOSE = """The meta-lesson is that a deployment decision is a set of explicit trade-offs, not a score. Responsible AI means publishing the threshold, the subgroup gaps, the fairness metrics that conflict and the confounds we cannot rule out, and letting accountable people choose with that information in front of them."""

def wc(s): return len(re.findall(r"\b[\w'$%.\-]+\b", s))
for k,v in ANS.items(): print(f"Q{k}: {wc(v)} words")
print("P4:", wc(P4_DIRECT), wc(P4_WHY), wc(P4_EVID), wc(P4_CLOSE), "total", wc(P4_DIRECT)+wc(P4_WHY)+wc(P4_EVID)+wc(P4_CLOSE))

Q1: 268 words
Q2: 287 words
Q3: 263 words
Q4: 281 words
Q5: 293 words
Q6: 257 words
Q7: 252 words
Q8: 279 words
P4: 50 354 56 49 total 509


In [12]:
# ---------------- PDF assembly ----------------
import os
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
FONT, FONTB = 'Helvetica', 'Helvetica-Bold'
for d in ['/usr/share/fonts/truetype/dejavu','/usr/share/fonts/dejavu']:
    if os.path.exists(f'{d}/DejaVuSans.ttf'):
        pdfmetrics.registerFont(TTFont('DV', f'{d}/DejaVuSans.ttf')); pdfmetrics.registerFont(TTFont('DVB', f'{d}/DejaVuSans-Bold.ttf'))
        FONT, FONTB = 'DV', 'DVB'; break
ss = getSampleStyleSheet()
BODY = ParagraphStyle('b', parent=ss['BodyText'], fontName=FONT, fontSize=9.6, leading=13.4, spaceAfter=6)
H1 = ParagraphStyle('h1', parent=ss['Heading1'], fontName=FONTB, fontSize=15, spaceBefore=10, spaceAfter=8, textColor=colors.HexColor('#1f4e79'))
H2 = ParagraphStyle('h2', parent=ss['Heading2'], fontName=FONTB, fontSize=11.5, spaceBefore=8, spaceAfter=5, textColor=colors.HexColor('#1f4e79'))
H3 = ParagraphStyle('h3', parent=ss['Heading3'], fontName=FONTB, fontSize=9.8, spaceBefore=6, spaceAfter=3)
CELL = ParagraphStyle('c', parent=BODY, fontSize=8, leading=10, spaceAfter=0)
CELLB = ParagraphStyle('cb', parent=CELL, fontName=FONTB)
TITLE = ParagraphStyle('t', parent=ss['Title'], fontName=FONTB, fontSize=19, textColor=colors.HexColor('#1f4e79'))
def P(t, st=BODY): return Paragraph(t, st)
def tbl(data, widths=None, head=1, fs=8):
    d = [[Paragraph(str(c), CELLB if i < head else CELL) for c in row] for i,row in enumerate(data)]
    t = Table(d, colWidths=widths, repeatRows=head)
    t.setStyle(TableStyle([('BACKGROUND',(0,0),(-1,head-1),colors.HexColor('#dbe7f3')),('GRID',(0,0),(-1,-1),.4,colors.grey),
        ('VALIGN',(0,0),(-1,-1),'MIDDLE'),('TOPPADDING',(0,0),(-1,-1),2.5),('BOTTOMPADDING',(0,0),(-1,-1),2.5)]))
    return KeepTogether([t])
def img(path, w=13.5*CM):
    from PIL import Image as PI
    iw, ih = PI.open(path).size; return Image(path, width=w, height=w*ih/iw)
S = []
S += [P("CSET485 - AI and Society: Assignment 1", TITLE),
      P("<b>Stress-testing a predictive model on the Adult Census Income dataset</b><br/>Roll No. 0769 (last digit 9 = Adult Census Income; random_state = 769; 70/30 train/test split)<br/>Submission deadline: 19 Oct 2026, 23:59 IST. Code: Jupyter notebook <i>CSET485_Assignment1.ipynb</i> (every number below is regenerated by running it top to bottom).", BODY)]

# ---- Preprocessing
S += [P("Data Preprocessing", H1),
 P(f"The Adult dataset (UCI adult.data + adult.test) has {n(R['n_rows'])} rows and 14 features; {R['base_rate']*100:.2f}% of people earn more than $50K. The UCI host was not reachable from my environment, so the identical data was read from a GitHub mirror (jbrownlee/Datasets, adult-all.csv); a local copy is shipped with the notebook. Target: income > $50K = 1, otherwise 0. For the sensitive-attribute analysis, sex is used (Group A = Female, Group B = Male)."),
 tbl([["Issue","What I found","Handling and justification"],
  ["Missing values", f"'?' appears only in workclass ({n(R['missing']['workclass'])}, {R['missing_pct']['workclass']}%), occupation ({n(R['missing']['occupation'])}, {R['missing_pct']['occupation']}%) and native-country ({n(R['missing']['native_country'])}, {R['missing_pct']['native_country']}%).", "Kept every row; recoded to an explicit category 'Unknown'. Missingness is plausibly informative (e.g. unemployed or never-worked), and dropping about 7% of rows could bias the sample."],
  ["Categorical encoding","workclass, marital_status, occupation, race, native_country (sex and married are binary).","One-hot encoding with drop_first=True to avoid the dummy-variable trap. Native country (40+ levels, about 90% US) collapsed to US / Other / Unknown. Final design matrix: " + str(R['n_features']) + " columns."],
  ["Feature scaling","Numeric features differ widely in scale (age 17-90, hours 1-99).","StandardScaler fitted on the TRAIN split only and applied to test, to avoid leakage. Scaling is applied to age, education_num, capital_gain, capital_loss, hours (used in the train/test models; the inferential models in Part 2.1-2.2 use unscaled values so coefficients are per-unit)."],
  ["Outliers", f"capital_gain is highly skewed and top-coded at 99,999 ({n(R['cap_gain_99999'])} rows); only {R['cap_gain_nonzero_pct']:.1f}% have non-zero gains and {R['cap_loss_nonzero_pct']:.1f}% non-zero losses. hours_per_week reaches 99.","capital_gain and capital_loss transformed with log1p. hours_per_week winsorised at the 1st/99th percentiles ({} and {} hours). Age and education left as recorded (bounded, plausible).".format(*[int(x) for x in R['hours_clip']])],
  ["Dropped columns","education duplicates education_num; fnlwgt is a survey sampling weight; relationship is collinear with marital_status and sex.","education, fnlwgt and relationship removed. Duplicate rows ("+str(R['duplicates'])+") were retained because Adult has no unique ID and identical records can be different people."],
  ["Split",f"Train {n(R['n_train'])} / test {n(R['n_test'])}.","train_test_split(test_size=0.30, random_state=769, stratify=y). Test-set base rate " + f"{R['test_base_rate']*100:.1f}%."]],
  widths=[2.6*CM,6.0*CM,8.4*CM])]

# ---- Part 1
S += [P("Part 1 - Build: Model A (OLS) versus Model B (logistic)", H1),
 P(f"Model A is an OLS linear probability model on the 0/1 income indicator (train R-squared = {p1['r2_A']:.3f}); its test predictions are cut at their own median ({p1['median_A']:.4f}) to give 'high risk'. Model B is a logistic regression cut at 0.5. Both were fitted on the same {n(R['n_train'])} training rows and evaluated on the {n(R['n_test'])} test rows."),
 tbl([["Confusion matrix (counts)","Model B: low risk (p < 0.5)","Model B: high risk (p >= 0.5)","Row total"],
      ["Model A: low risk (< median)", n(cmx[0][0]), n(cmx[0][1]), n(sum(cmx[0]))],
      ["Model A: high risk (>= median)", n(cmx[1][0]), n(cmx[1][1]), n(sum(cmx[1]))],
      ["Column total", n(cmx[0][0]+cmx[1][0]), n(cmx[0][1]+cmx[1][1]), n(R['n_test'])]], widths=[5.2*CM,4.2*CM,4.2*CM,3.4*CM]),
 Spacer(1,6),
 tbl([["Metric","Model A (median rule)","Model B (0.5 rule)"],
      ["Flagged high risk", f"{n(p1['flagged_A'])} ({p1['flagged_A_pct']:.1f}%)", f"{n(p1['flagged_B'])} ({p1['flagged_B_pct']:.1f}%)"],
      ["Accuracy vs true label", f"{p1['acc_A']*100:.1f}%", f"{p1['acc_B']*100:.1f}%"],
      ["Precision / recall", f"{p1['prec_A']*100:.1f}% / {p1['rec_A']*100:.1f}%", f"{p1['prec_B']*100:.1f}% / {p1['rec_B']*100:.1f}%"],
      ["Mismatched individuals", f"{n(p1['mismatch'])} of {n(R['n_test'])} = {p1['mismatch_pct']:.2f}%", f"(A-only {n(cmx[1][0])}; B-only {cmx[0][1]})"]], widths=[5.2*CM,6*CM,5.8*CM]),
 Spacer(1,6), img('figs/ols_hist.png', 11.5*CM),
 P(f"<b>Why do they differ?</b> The two rules use different cut-offs. The median rule is a relative rule that always flags 50% of people (median OLS score {p1['median_A']:.3f}); the logistic rule is an absolute probability rule that flags {p1['flagged_B_pct']:.1f}%, close to the {R['test_base_rate']*100:.1f}% base rate. OLS on a binary target is also unbounded ({p1['frac_A_below0']:.1f}% of predictions are below 0; range {p1['min_A']:.2f} to {p1['max_A']:.2f}). Cutting OLS at 0.5 instead cuts the mismatch to {p1['A_cut05_vs_B_mismatch_pct']:.2f}%. Full discussion in Part 3, Q1.")]

# ---- Part 2.1
S += [PageBreak(), P("Part 2 - The Trap", H1), P("2.1 Simpson's paradox hunt (split by marital status)", H2),
 P(f"Split variable: married-civ-spouse (n = {n(p21['n_married'])}) versus all other marital statuses (n = {n(p21['n_unmarried'])}). Logistic regression (statsmodels) on the full dataset ({n(p21['n_full'])} rows) and each subgroup, same controls, marital-status dummies dropped in the subgroups. Coefficients are unscaled logit coefficients."),
 tbl([["Predictor","Full model (95% CI)","Married subgroup (95% CI)","Not-married subgroup (95% CI)"]] +
     [[nm, ci(p21[k]['full']), ci(p21[k]['married']), ci(p21[k]['unmarried'])] for k,nm in [('male','<b>male (key predictor)</b>'),('age','age'),('education_num','education_num'),('hours','hours'),('capital_gain','capital_gain (log1p)')]],
     widths=[3.4*CM,4.5*CM,4.5*CM,4.6*CM]),
 Spacer(1,6), img('figs/simpson.png', 10.5*CM),
 tbl([["Raw >50K rate","Female","Male"],["Overall",f"{cr['female_rate']*100:.1f}%",f"{cr['male_rate']*100:.1f}%"],["Married-civ-spouse",f"{cr['female_married']*100:.1f}%",f"{cr['male_married']*100:.1f}%"],["Not married",f"{cr['female_unmarried']*100:.1f}%",f"{cr['male_unmarried']*100:.1f}%"],["Share who are married",f"{cr['female_share_married']:.1f}%",f"{cr['male_share_married']:.1f}%"]], widths=[5*CM,3*CM,3*CM]),
 P(f"<b>Result.</b> The sign of the male coefficient reverses: {male['full']['coef']:+.3f} overall, {male['married']['coef']:+.3f} among married people, {male['unmarried']['coef']:+.3f} among the unmarried. The male-by-married interaction is {p21['interaction']['coef']:+.3f} (CI {p21['interaction']['lo']:+.3f} to {p21['interaction']['hi']:+.3f}). Age, hours and capital-gain slopes also differ in magnitude (e.g. hours {p21['hours']['married']['coef']:.3f} vs {p21['hours']['unmarried']['coef']:.3f}). <b>Why:</b> marital status is both the strongest predictor of high income ({cr['married_rate']*100:.1f}% vs {cr['unmarried_rate']*100:.1f}%) and unevenly distributed by sex ({cr['male_share_married']:.1f}% of men vs {cr['female_share_married']:.1f}% of women are married), so the pooled coefficient blends two opposite conditional effects. <b>Policy:</b> ignoring the subgroup would suggest a small, uniform male advantage (odds ratio {male['full']['odds']:.2f}); in fact unmarried men have {male['unmarried']['odds']:.2f} times the odds of unmarried women, while married men have {male['married']['odds']:.2f} times the odds of married women. See Q2.")]

# ---- Part 2.2
S += [P("2.2 Omitted-variable bias (dropped variable: marital_status)", H2),
 tbl([["Predictor","Full model coef","Reduced model coef","% change"]] + [[r['var'], f"{r['full']:+.4f}", f"{r['reduced']:+.4f}", f"{r['pct']:+.1f}%"] for r in p22['rows']], widths=[4*CM,4*CM,4*CM,3.5*CM]),
 P(f"<b>Largest shift: male</b> ({rows22['male']['full']:+.4f} to {rows22['male']['reduced']:+.4f}, {rows22['male']['pct']:+.0f}%; odds ratio {math.exp(rows22['male']['full']):.2f} to {math.exp(rows22['male']['reduced']):.2f}), followed by age ({rows22['age']['pct']:+.1f}%). Correlations of the dropped marital indicator with included predictors: male {p22['corr_married']['male']:.2f}, age {p22['corr_married']['age']:.2f}, hours {p22['corr_married']['hours']:.2f}. The reduced model fits worse (pseudo-R-squared {p22['pr2_full']:.3f} to {p22['pr2_red']:.3f}; AIC +{p22['aic_red']-p22['aic_full']:,.0f}) but still looks 'significant', so a model that is not stress-tested for confounds can report a sex effect about {rows22['male']['reduced']/rows22['male']['full']:.0f} times too large. See Q3 and Q4 for detection in the real world.")]

# ---- Part 2.3
pf = lambda d,k: f"{d[k]:.1f}" if d else "n/a"
S += [P("2.3 Adversarial subset (confident and wrong)", H2),
 P(f"Model B on the test set: {n(p23['n_fp'])} confident false positives (p >= 0.75, y = 0) and {n(p23['n_fn'])} confident false negatives (p <= 0.25, y = 1) = <b>{n(p23['n_bad'])} cases = {p23['pct_bad']:.2f}%</b> of the test set ({p23['pct_bad_of_conf']:.1f}% of the {n(p23['n_conf'])} confident predictions; FP rate among p >= 0.75: {p23['fp_rate_hi']:.1f}%; FN rate among p <= 0.25: {p23['fn_rate_lo']:.1f}%). A random sample of 50 (seed 769: {p23['sample_fp']} FP, {p23['sample_fn']} FN) is saved as adversarial_50_rows.csv."),
 tbl([["Characteristic","50-row sample","of which FP","of which FN","Full test set"]] +
   [[lab, f"{sa[k]:.1f}", pf(fpP,k), pf(fnP,k), f"{sb[k]:.1f}"] for lab,k in [("Mean age","age_mean"),("% aged 50+","pct_age_ge50"),("% aged under 30","pct_age_lt30"),("% male","pct_male"),("% married-civ-spouse","pct_married"),("Mean years of education","edu_mean"),("% bachelor's or higher","pct_bach_plus"),("Mean hours per week","hours_mean"),("% working over 45 hours","pct_hours_gt45"),("% white","pct_white"),("% professional/managerial","pct_prof"),("% with capital gain","pct_cap_gain")]],
   widths=[5*CM,3*CM,3*CM,3*CM,3*CM]),
 P(f"<b>Common characteristic:</b> {sa['pct_married']:.0f}% of the 50 rows are married-civ-spouse versus {sb['pct_married']:.1f}% of the test set (binomial p = {p23['binom_p_married']:.1e}); 100% of the false positives are married, mostly highly educated, long-hours men aged about {fpP['age_mean']:.0f}. <b>Why:</b> the logistic model adds a single constant ({ct['marital_status_Married-civ-spouse']:+.2f} log-odds) for marriage and stacks education, hours and age on top, but marriage interacts with sex, hours and age (Part 2.1) and age has an inverted-U relationship with income that a linear term cannot represent. See Q5.")]

# ---- Part 2.4
def row(r): return [f"{r['t']:.2f}", f(r['DP']), f(r['EO']), f(r['PP']), f(r['TPRr']), f(r['FPRr']), f"{r['sel_A']*100:.1f}% / {r['sel_B']*100:.1f}%", f"{r['PPV_A']*100:.1f}% / {r['PPV_B']*100:.1f}%", f"{r['acc']*100:.1f}%"]
S += [PageBreak(), P("2.4 Fairness trade-off analysis (sensitive variable: sex; Group A = Female, Group B = Male)", H2),
 P(f"Logistic model B on the test set (n = {n(R['n_test'])}: {n(p24['nA'])} women, {n(p24['nB'])} men; positive rates {p24['base_A']*100:.1f}% and {p24['base_B']*100:.1f}%). Ratios are A/B. 'n/a' means an undefined ratio (denominator zero)."),
 tbl([["t","DP","EO","PP","TPR ratio","FPR ratio","Selected F / M","Precision F / M","Accuracy"]] + [row(r) for r in p24['rows']], widths=[1.1*CM,1.5*CM,1.5*CM,1.5*CM,1.8*CM,1.8*CM,3.0*CM,3.2*CM,1.8*CM]),
 Spacer(1,6), img('figs/tradeoff.png', 10.5*CM), img('figs/metrics_vs_t.png', 10*CM),
 P(f"<b>Quantifying the trade-off.</b> DP and EO are almost perfectly correlated across thresholds (r = {p24['dp_eo_corr']:.3f}): both are best at t = 0.05 and degrade as t rises, so in this model there is <i>no</i> DP-versus-EO trade-off to measure along the threshold axis (the one local reversal, EO rising from {f(sw[0.75]['EO'])} to {f(sw[0.85]['EO'])} between t = 0.75 and 0.85, rests on a handful of selected women and is noise). The genuine trade-off is between DP/EO and predictive parity and accuracy. Moving from t = {tr['frm']:.2f} to t = {tr['to']:.2f}: <b>if you improve DP from {f(t45['DP'])} to {f(t05['DP'])} (a {tr['dp_gain']:.0f}% gain), PP drops from {f(t45['PP'])} to {f(t05['PP'])} (a {abs(tr['pp_change']):.0f}% loss); the trade-off cost is {tr['cost_dp_pp']:.2f}</b>. Likewise EO improves from {f(t45['EO'])} to {f(t05['EO'])} ({tr['eo_gain']:.0f}% gain) for the same PP loss (cost {tr['cost_eo_pp']:.2f}) and accuracy falls by {abs(tr['acc_change']):.0f}% (from {t45['acc']*100:.1f}% to {t05['acc']*100:.1f}%)."),
 P(f"<b>Key insight.</b> No threshold satisfies all three: PP is closest to 1 at t = {p24['best_t']['PP']:.2f} ({f(sw[p24['best_t']['PP']]['PP'])}) where DP is only {f(sw[p24['best_t']['PP']]['DP'])} and EO {f(sw[p24['best_t']['PP']]['EO'])}; DP and EO are best at t = 0.05 where PP is {f(t05['PP'])}. I would recommend t = 0.35 as the default operating point (PP {f(t35['PP'])}, accuracy {t35['acc']*100:.1f}%) with human review, and t = 0.15 as the equal-opportunity-leaning alternative (TPR ratio {f(t15['TPRr'])} versus {f(t35['TPRr'])}, EO {f(t15['EO'])}, at a PP of {f(t15['PP'])} and accuracy of {t15['acc']*100:.1f}%). Neither is 'fair' by all three measures; the choice is a values decision (Q6-Q8).")]

# ---- Part 3
QS = {1:"Why did the threshold-based approach and the direct logistic approach flag (or not flag) the same individuals?",
 2:"The subgroup with differing effects: coefficients, confidence intervals, policy conclusion.",
 3:"Omitted-variable test: dropped variable, coefficient shifts, and what it says about trusting unstressed models.",
 4:"As a practitioner, how would you detect a hidden confound you cannot currently see?",
 5:"The 50-row adversarial subset: shared characteristic and failure mechanism.",
 6:"What does the trade-off cost mean for a real person? A two-person scenario.",
 7:"Defending a specific threshold with at least three numbers, and the strongest counterargument.",
 8:"Demographic parity or equalized odds for this dataset's context?"}
S += [PageBreak(), P("Part 3 - Descriptive Analysis Questions", H1)]
for k in range(1,9):
    S += [P(f"Q{k}. {QS[k]}", H3), P(ANS[k])]
    if k == 3:
        S += [tbl([["Predictor","Before (full)","After (marital_status dropped)","% change"]] + [[r['var'], f"{r['full']:+.4f}", f"{r['reduced']:+.4f}", f"{r['pct']:+.1f}%"] for r in p22['rows']], widths=[4*CM,3.8*CM,5*CM,3*CM]), Spacer(1,6)]
    if k == 6:
        S += [tbl([["Threshold","TPR F / M","FPR F / M","Precision F / M","Selected F / M","Accuracy"]] + [[f"{t:.2f}", f"{sw[t]['TPR_A']*100:.1f}% / {sw[t]['TPR_B']*100:.1f}%", f"{sw[t]['FPR_A']*100:.1f}% / {sw[t]['FPR_B']*100:.1f}%", f"{sw[t]['PPV_A']*100:.1f}% / {sw[t]['PPV_B']*100:.1f}%", f"{sw[t]['sel_A']*100:.1f}% / {sw[t]['sel_B']*100:.1f}%", f"{sw[t]['acc']*100:.1f}%"] for t in (0.05,0.35,0.45)], widths=[2*CM,3*CM,3*CM,3.2*CM,3.2*CM,2.2*CM]), Spacer(1,6)]

# ---- Part 4
S += [PageBreak(), P("Part 4 - Reflection: 'Just give me one number: deploy or not?'", H1),
 P("<b>1. Direct answer.</b> " + P4_DIRECT), P("<b>2. Why a single number fails.</b> " + P4_WHY), P("<b>3. Grounding in evidence.</b> " + P4_EVID), P("<b>4. Closing.</b> " + P4_CLOSE),
 Spacer(1,10), P("<b>Reproducibility.</b> The notebook CSET485_Assignment1.ipynb reads adult-all.csv, fixes random_state = 769, writes results.json, figures and adversarial_50_rows.csv, and regenerates this PDF. Python 3, pandas, numpy, scikit-learn, statsmodels, scipy, matplotlib, reportlab.", BODY)]

def footer(c, d):
    c.saveState(); c.setFont(FONT, 7.5); c.drawString(2*CM, 1.1*CM, "CSET485 Assignment 1 - Roll 0769 - Adult Census Income"); c.drawRightString(A4[0]-2*CM, 1.1*CM, f"Page {d.page}"); c.restoreState()
doc = SimpleDocTemplate('CSET485_Assignment1_Report.pdf', pagesize=A4, leftMargin=2*CM, rightMargin=2*CM, topMargin=1.8*CM, bottomMargin=1.8*CM, title='CSET485 Assignment 1')
doc.build(S, onFirstPage=footer, onLaterPages=footer)
for k,v in ANS.items(): print(f"Q{k}: {wc(v)} words")
print('P4 total', wc(P4_DIRECT)+wc(P4_WHY)+wc(P4_EVID)+wc(P4_CLOSE)); print('PDF built')

Q1: 268 words
Q2: 287 words
Q3: 263 words
Q4: 281 words
Q5: 293 words
Q6: 257 words
Q7: 252 words
Q8: 279 words
P4 total 509
PDF built
